# Prétraitement des surfaces en GCPE

L'exécution de ce script suppose d'avoir déjà effectué l'ensemble des étapes dans le [README.md](./README.md)

Dépendances : 
- [surface_departementales_brut_COP.csv](./brut/surface_departementales_brut_COP.csv)
- [surface_departementales_brut_FOU.csv](./brut/surface_departementales_brut_FOU.csv)
- [surface_departementales_brut_IND.csv](./brut/surface_departementales_brut_IND.csv)
- [surface_departementales_brut_PDT.csv](./brut/surface_departementales_brut_PDT.csv)

Ce notebook permet de lire les fichiers bruts et de compiler un unique fichier de sorite [surface_espece_ancienne_region](./surface_espece_ancienne_region.csv) disposant des colonnes suivante : 
- Espece_SSP
- Campagne
- Nom_Ancienne_Region
- Surface_Espece_Region
- Surface_Espece_Region_sans_prairie
- Surface_Region
- Part_surface_espece_region_sans_prairie
- Part_surface_espece_region


## Import des librairies

In [171]:
import pandas as pd
from tqdm import tqdm
import numpy as np

## Paramètres globaux

In [186]:
path_file = './'

ENTREPOT_PATH = '~/Bureau/utils/data/'
df = {}
STUDIED_YEAR = 2025
# Attention, Nicolas Chartier suggérait d'utiliser plutôt les tableaux interractifs : https://agreste.agriculture.gouv.fr/agreste-saiku/?plugin=true&query=query/open/SAANR_DEVELOPPE_2#query/open/SAANR_DEVELOPPE_2
FILE_BRUT_PATH = './brut/'
PREFIX_FILE_BRUT = 'surface_departementales_brut_'
FILES = [
    {
        'name' : 'Cultures principales',
        'abreviation' : 'COP',
        'matching_dict' : {
            "03 - Total blé tendre (01 + 02)" : "Blé tendre",
            "06 - Total blé dur (04 + 05)" : "Blé dur",
            "08 - Orge d'hiver et escourgeon" : "Orge d'hiver",
            "09 - Orge de printemps" : "Orge de printemps",
            "18 - Total maïs grain et maïs semence (16 + 17 )" : "Maïs grain",
            "20 - Triticale" : "Triticale",
            "31 - Total colza grain et navette (29 + 30)" : "Colza",
            "32 - Tournesol" : "Tournesol",
            "41 - Pois protéagineux" : "Pois protéagineux",
        }
    },
    {
        'name' : 'Cultures industrielles',
        'abreviation' : 'IND',
        'matching_dict' : {
            "01 - Betterave industrielle" : "Betterave sucrière",
            "05 - Lin textile" : "Lin fibre",
            "02 - Canne à sucre" : "Canne à sucre"
        }
    },
    {
        'name' : 'Cultures fourragères',
        'abreviation' : 'FOU',
        'matching_dict' : {
            "06 - Total maïs fourrage et ensilage (04 + 05)" : "Maïs fourrage",
            "13 - Total prairies non permanentes (11 + 12)" : "Prairies non permanentes"
        }
    },
    {
        'name' : 'Cultures racines',
        'abreviation' : 'PDT',
        'matching_dict' : {
            "07 - Pommes de terre (01 + … + 05)" : "Pomme de terre"
        }
    },
]

DICT_MATCH_DEPT_ANCIENNE_REGION = {
    # Auvergne
    '003': 'Auvergne',
    '015': 'Auvergne',
    '043': 'Auvergne',
    '063': 'Auvergne',

    # Basse-Normandie
    '014': 'Basse-Normandie',
    '050': 'Basse-Normandie',
    '061': 'Basse-Normandie',

    # Bourgogne
    '021': 'Bourgogne',
    '058': 'Bourgogne',
    '071': 'Bourgogne',
    '089': 'Bourgogne',

    # Bretagne
    '022': 'Bretagne',
    '029': 'Bretagne',
    '035': 'Bretagne',
    '056': 'Bretagne',

    # Centre
    '018': 'Centre-Val de Loire',
    '028': 'Centre-Val de Loire',
    '036': 'Centre-Val de Loire',
    '037': 'Centre-Val de Loire',
    '041': 'Centre-Val de Loire',
    '045': 'Centre-Val de Loire',

    # Champagne-Ardenne
    '008': 'Champagne-Ardenne',
    '010': 'Champagne-Ardenne',
    '051': 'Champagne-Ardenne',
    '052': 'Champagne-Ardenne',

    # Corse
    '2A': 'Corse',
    '2B': 'Corse',

    # Franche-Comté
    '025': 'Franche-Comté',
    '039': 'Franche-Comté',
    '070': 'Franche-Comté',
    '090': 'Franche-Comté',

    # Haute-Normandie
    '027': 'Haute-Normandie',
    '076': 'Haute-Normandie',

    # Île-de-France
    '075': 'Île-de-France',
    '077': 'Île-de-France',
    '078': 'Île-de-France',
    '091': 'Île-de-France',
    '092': 'Île-de-France',
    '093': 'Île-de-France',
    '094': 'Île-de-France',
    '095': 'Île-de-France',

    # Languedoc-Roussillon
    '011': 'Languedoc-Roussillon',
    '030': 'Languedoc-Roussillon',
    '034': 'Languedoc-Roussillon',
    '048': 'Languedoc-Roussillon',
    '066': 'Languedoc-Roussillon',

    # Limousin
    '019': 'Limousin',
    '023': 'Limousin',
    '087': 'Limousin',

    # Lorraine
    '054': 'Lorraine',
    '055': 'Lorraine',
    '057': 'Lorraine',
    '088': 'Lorraine',

    # Midi-Pyrénées
    '009': 'Midi-Pyrénées',
    '012': 'Midi-Pyrénées',
    '031': 'Midi-Pyrénées',
    '032': 'Midi-Pyrénées',
    '046': 'Midi-Pyrénées',
    '065': 'Midi-Pyrénées',
    '081': 'Midi-Pyrénées',
    '082': 'Midi-Pyrénées',

    # Nord-Pas-de-Calais
    '059': 'Nord-Pas-de-Calais',
    '062': 'Nord-Pas-de-Calais',

    # Pays de la Loire
    '044': 'Pays de la Loire',
    '049': 'Pays de la Loire',
    '053': 'Pays de la Loire',
    '072': 'Pays de la Loire',
    '085': 'Pays de la Loire',

    # Picardie
    '002': 'Picardie',
    '060': 'Picardie',
    '080': 'Picardie',

    # Poitou-Charentes
    '016': 'Poitou-Charentes',
    '017': 'Poitou-Charentes',
    '079': 'Poitou-Charentes',
    '086': 'Poitou-Charentes',

    # Provence-Alpes-Côte d'Azur
    '004': 'Provence-Alpes-Côte d’Azur',
    '005': 'Provence-Alpes-Côte d’Azur',
    '006': 'Provence-Alpes-Côte d’Azur',
    '013': 'Provence-Alpes-Côte d’Azur',
    '083': 'Provence-Alpes-Côte d’Azur',
    '084': 'Provence-Alpes-Côte d’Azur',

    # Rhône-Alpes
    '001': 'Rhône-Alpes',
    '007': 'Rhône-Alpes',
    '026': 'Rhône-Alpes',
    '038': 'Rhône-Alpes',
    '042': 'Rhône-Alpes',
    '069': 'Rhône-Alpes',
    '073': 'Rhône-Alpes',
    '074': 'Rhône-Alpes',

    # Alsace
    '067': 'Alsace',
    '068': 'Alsace',

    # Aquitaine
    '024': 'Aquitaine',
    '033': 'Aquitaine',
    '040': 'Aquitaine',
    '047': 'Aquitaine',
    '064': 'Aquitaine',

    # DOM / régions d'outre-mer
    '971': 'Guadeloupe',
    '972': 'Martinique',
    '973': 'Guyane',
    '974': 'La Réunion',
    '976': 'Mayotte',
}

## Chargement de toutes les données

In [173]:
# on compile un dataframe avec toutes les données qui nous intéressent
res = []
for file in FILES :
    file_path = FILE_BRUT_PATH+PREFIX_FILE_BRUT+file['abreviation']+'.csv'
    df[PREFIX_FILE_BRUT+file['abreviation']] = pd.read_csv(file_path)
    
    # premiers pretraitements
    columns = df[PREFIX_FILE_BRUT+file['abreviation']].iloc[4]
    df[PREFIX_FILE_BRUT+file['abreviation']] = df[PREFIX_FILE_BRUT+file['abreviation']].iloc[5:]

    # mise à jour des colonnes
    df[PREFIX_FILE_BRUT+file['abreviation']].columns = columns

    res.append(df[PREFIX_FILE_BRUT+file['abreviation']])


df[PREFIX_FILE_BRUT+'complet'] = pd.concat(res)

## Nettoyage préalable des données
> Attention, si le format d'entrée des données Agreste évolue, ces étapes pourraient ne plus être exactement valides.

In [174]:
df[PREFIX_FILE_BRUT+'complet']['numero_departement'] = df[PREFIX_FILE_BRUT+'complet']['LIB_DEP'].str.split(' - ').apply(lambda x: x[0])

## Traitement spécifique à surface_culture_departementales_agreste

In [175]:
# on compile un dictionnaire de matching des cultures complet
CULTURE_MATCHING_DICT = {}
for file in FILES : 
    CULTURE_MATCHING_DICT = CULTURE_MATCHING_DICT | file['matching_dict']

In [176]:
# création du dataframe permettant d'effectuer le match entre nomenclature Agreste et CAN
df['match_agreste_can'] = pd.DataFrame([CULTURE_MATCHING_DICT]).T.reset_index().rename(columns={
    0:"nomenclature_can",
    'index' : "nomenclature_agreste"
})

# reclassification des cultures selon la nomenclature CAN
left = df[PREFIX_FILE_BRUT+'complet']
right = df['match_agreste_can']
df[PREFIX_FILE_BRUT+'extanded'] = pd.merge(left, right, left_on ='LIB_SAA', right_on='nomenclature_agreste')

# création du dataframe permettant d'effectuer le match entre les départements et les noms des anciennes région
df['match_departement_ancienne_region'] = pd.DataFrame([DICT_MATCH_DEPT_ANCIENNE_REGION]).T.reset_index().rename(columns={
    0:"nom_ancienne_region",
    'index' : "numero_departement"
})

# reclassification des departements.
left = df[PREFIX_FILE_BRUT+'extanded']
right = df['match_departement_ancienne_region']
df[PREFIX_FILE_BRUT+'extanded'] = pd.merge(left, right, on='numero_departement')

In [177]:
LAST_DISPONIBLE_YEAR = 2025
YEARS = [k for k in range(2011, LAST_DISPONIBLE_YEAR+1)]

In [178]:
df['numero_departement_espece_surface'] = pd.melt(
        df[PREFIX_FILE_BRUT+'extanded'],
        id_vars=['numero_departement', 'nom_ancienne_region', 'nomenclature_can'],
        value_vars=["SURF_"+str(year) for year in YEARS]      
).rename(columns={
    'variable' : 'nom_variable',
    'value' : 'surface'
})
# obtention de la campagne dans un champ int
df['numero_departement_espece_surface']['campagne'] = df['numero_departement_espece_surface']['nom_variable'].str.split('_').apply(lambda x: x[1]).astype(int)
# obtention de la surface
df['numero_departement_espece_surface']['surface'] = df['numero_departement_espece_surface']['surface'].str.replace(' ', '').fillna(0).astype('int')

# exclusion des "sommes"
df['numero_departement_espece_surface'] = df['numero_departement_espece_surface'].loc[
    (df['numero_departement_espece_surface']['nomenclature_can'] != 'somme') 
    & (df['numero_departement_espece_surface']['nomenclature_can'] != 'null')
]

# groupement 
df[PREFIX_FILE_BRUT+'final'] = df['numero_departement_espece_surface'].groupby(['nom_ancienne_region', 'campagne', 'nomenclature_can']).agg({
    'surface' : 'sum'
}).reset_index()

## Global

In [179]:
df[PREFIX_FILE_BRUT+'final']

,nom_ancienne_region,campagne,nomenclature_can,surface
0,Alsace,2011,Betterave sucrière,6219
1,Alsace,2011,Blé dur,0
2,Alsace,2011,Blé tendre,48590
3,Alsace,2011,Canne à sucre,0
4,Alsace,2011,Colza,3595
...,...,...,...,...
5845,Île-de-France,2025,Pois protéagineux,6309
5846,Île-de-France,2025,Pomme de terre,5147
5847,Île-de-France,2025,Prairies non permanentes,24023
5848,Île-de-France,2025,Tournesol,6580


In [180]:
# calcul des surfaces totales disponibles pour les régions (on compte tout : y compris les prairies)
df['surface_ancienne_region_avec_prairie'] = df[PREFIX_FILE_BRUT+'final'].groupby(['nom_ancienne_region', 'campagne']).agg({
    'surface' : 'sum'
}).reset_index()

# calcul des surfaces totales sans les prairies (nécessaires pour calculer un ift de référence sans prairies)
df['surface_ancienne_region_prairie'] = df[PREFIX_FILE_BRUT+'final'].loc[
    ~df[PREFIX_FILE_BRUT+'final']['nomenclature_can'].isin(
        ['Prairies non permanentes']
    )
].groupby(['nom_ancienne_region', 'campagne']).agg({
    'surface' : 'sum'
}).reset_index().rename(columns = {'surface' : 'surface_sans_prairie'})

df['surface_ancienne_region']  = pd.merge(
    df['surface_ancienne_region_avec_prairie'], 
    df['surface_ancienne_region_prairie'],
    left_on = ['nom_ancienne_region', 'campagne'],
    right_on = ['nom_ancienne_region', 'campagne'], 
    how='left'
)

In [181]:
# ajout des surfaces totales obtenues
left = df[PREFIX_FILE_BRUT+'final']
right = df['surface_ancienne_region'].rename(columns={'surface' : 'surface_ancienne_region', 'surface_sans_prairie' : 'surface_ancienne_region_sans_prairie'})
df['ancienne_region_espece_surface_extanded'] = pd.merge(left, right, on=['nom_ancienne_region', 'campagne'], how='left')

# calcul du pourcentage de surface pour la culture
df['ancienne_region_espece_surface_extanded']['prc_surface'] = round(df['ancienne_region_espece_surface_extanded']['surface'] / df['ancienne_region_espece_surface_extanded']['surface_ancienne_region'] * 100, 2)
df['ancienne_region_espece_surface_extanded']['prc_sans_prairie_surface'] = round(df['ancienne_region_espece_surface_extanded']['surface'] / df['ancienne_region_espece_surface_extanded']['surface_ancienne_region_sans_prairie'] * 100, 2)

In [182]:
df['final'] = df['ancienne_region_espece_surface_extanded'].rename(columns={
    'nom_ancienne_region' : 'Nom_Ancienne_Region',
    'campagne' : 'Campagne', 
    'surface':  'Surface_Espece_Region',
    'nomenclature_can' : 'Espece_SSP',
    'surface_ancienne_region' : 'Surface_Region', 
    'surface_ancienne_region_sans_prairie' : 'Surface_Region_sans_prairie', 
    'prc_surface' : 'Part_surface_espece_region',
    'prc_sans_prairie_surface' : 'Part_surface_espece_region_sans_prairie' 
})[[
    'Espece_SSP', 'Campagne', 'Nom_Ancienne_Region', 
    'Surface_Espece_Region', 'Surface_Region',  'Surface_Region_sans_prairie',
    'Part_surface_espece_region', 'Part_surface_espece_region_sans_prairie'
]].sort_values([
     'Campagne', 'Nom_Ancienne_Region',
])

In [183]:
df['final']

,Espece_SSP,Campagne,Nom_Ancienne_Region,Surface_Espece_Region,Surface_Region,Surface_Region_sans_prairie,Part_surface_espece_region,Part_surface_espece_region_sans_prairie
0,Betterave sucrière,2011,Alsace,6219,220090,210962,2.83,2.95
1,Blé dur,2011,Alsace,0,220090,210962,0.00,0.00
2,Blé tendre,2011,Alsace,48590,220090,210962,22.08,23.03
3,Canne à sucre,2011,Alsace,0,220090,210962,0.00,0.00
4,Colza,2011,Alsace,3595,220090,210962,1.63,1.70
...,...,...,...,...,...,...,...,...
5845,Pois protéagineux,2025,Île-de-France,6309,496527,472504,1.27,1.34
5846,Pomme de terre,2025,Île-de-France,5147,496527,472504,1.04,1.09
5847,Prairies non permanentes,2025,Île-de-France,24023,496527,472504,4.84,5.08
5848,Tournesol,2025,Île-de-France,6580,496527,472504,1.33,1.39


In [184]:
# vérification qu'on somme à peu près à 100 pour un cas précis.
df['test'] = df['final'].loc[
    (df['final']['Nom_Ancienne_Region'] == 'Bourgogne') & 
    (df['final']['Campagne'] == 2011)
]
df['test']['Part_surface_espece_region'].sum()

np.float64(99.98)

In [185]:
df['final'].to_csv('surface_espece_ancienne_region.csv', index=False)